# Notebook 5: Formula Validation (evidence for the thesis appendix)
This notebook **checks the formulas** used in the project with independent calculations. It produces one table of checks (`results/formula_validation.csv`) that you can put in the appendix.

| Check | What it proves |
|---|---|
| V1 | RMSE, MAE and MAPE written by hand give the same values as the library |
| V2 | The pricing rule, calculated by hand, gives the same price as the pricing engine (including floor, ceiling and rounding) |
| V3 | SHAP values add up to the model prediction |
| V4 | The LKR effect of each reason adds up to the demand effect on the price |
| V5 | The revenue simulation, recalculated independently, gives the same numbers as the saved results table |

## Step 1: Load

In [ ]:
import os, json, importlib.util, joblib
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pricing_project'
except ImportError:
    BASE = os.environ.get('PROJECT_DIR', '.')

MODELS, RESULTS = f'{BASE}/models', f'{BASE}/results'
df = pd.read_csv(f'{BASE}/data/clean/main_clean.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
FEATURES = joblib.load(f'{MODELS}/features.pkl')
xgb = joblib.load(f'{MODELS}/xgboost.pkl')
lr = joblib.load(f'{MODELS}/linear_regression.pkl')
cfg = json.load(open(f'{MODELS}/pricing_config.json'))
spec = importlib.util.spec_from_file_location('pe', f'{MODELS}/pricing_engine.py')
pe = importlib.util.module_from_spec(spec); spec.loader.exec_module(pe)

split = int(len(df) * 0.8)
test = df.iloc[split:].copy()
y = test.Historical_Occupancy_Rate.values
checks = []   # (id, check, hand calculation, library or system, difference, result)
print('Test days:', len(test), '| pricing parameters:', cfg)

## V1: Error metrics by hand
- RMSE = square root of the mean of (actual - predicted)^2
- MAE = mean of |actual - predicted|
- MAPE = mean of |actual - predicted| / actual x 100

In [ ]:
for name, model in [('Linear Regression', lr), ('XGBoost', xgb)]:
    p = model.predict(test[FEATURES])
    hand = {'RMSE': np.sqrt(np.mean((y - p) ** 2)),
            'MAE': np.mean(np.abs(y - p)),
            'MAPE': np.mean(np.abs(y - p) / y) * 100}
    lib = {'RMSE': np.sqrt(mean_squared_error(y, p)),
           'MAE': mean_absolute_error(y, p),
           'MAPE': mean_absolute_percentage_error(y, p) * 100}
    for m in hand:
        diff = abs(hand[m] - lib[m])
        checks.append(('V1', f'{name} {m}', round(hand[m], 4), round(lib[m], 4), diff, 'Pass' if diff < 1e-6 else 'Fail'))
pd.DataFrame(checks, columns=['id', 'check', 'by hand', 'library', 'difference', 'result'])

## V2: Pricing rule by hand (6 test cases)
Rule: demand price = B x (1 + K x (occupancy - 60) / 100); mixed price = W x demand price + (1 - W) x competitor price; limit to 0.9 B and 2.0 B; round to the nearest 50.
The hand calculation below is written **without** calling the pricing engine.

In [ ]:
K, W, NORMAL = cfg['K'], cfg['W'], cfg['normal_occupancy']
LO, HI, STEP = cfg['min_factor'], cfg['max_factor'], cfg['round_to']

def by_hand(B, occ, comp):
    demand_price = B * (1 + K * (occ - NORMAL) / 100)
    mixed = W * demand_price + (1 - W) * comp
    limited = min(max(mixed, LO * B), HI * B)
    return round(limited / STEP) * STEP, demand_price, mixed

cases = [
    ('Normal day, price = market', 8000, 60, 8000),
    ('High demand', 12000, 75, 15000),
    ('Very low demand (floor applies)', 12000, 30, 9000),
    ('Very high market price (ceiling applies)', 8000, 95, 30000),
    ('Rounding to 50', 10000, 52, 11000),
    ('Demand above market', 15000, 85, 14000),
]
rows = []
for label, B, occ, comp in cases:
    hand, dp, mixed = by_hand(B, occ, comp)
    sysv = pe.recommend_price(occ, B, comp)
    rows.append((label, B, occ, comp, round(dp), round(mixed), hand, sysv))
    checks.append(('V2', label, hand, sysv, abs(hand - sysv), 'Pass' if hand == sysv else 'Fail'))
pd.DataFrame(rows, columns=['case', 'base rate', 'occupancy %', 'competitor', 'demand price', 'mixed price', 'by hand', 'engine'])

## V3: SHAP values add up to the prediction
For every test day: expected value + sum of SHAP values = model prediction.

In [ ]:
explainer = shap.TreeExplainer(xgb)
sv = explainer.shap_values(test[FEATURES])
base_value = float(np.ravel(explainer.expected_value)[0])
recon = base_value + sv.sum(axis=1)
pred = xgb.predict(test[FEATURES])
err = np.abs(recon - pred).max()
checks.append(('V3', 'SHAP additivity (max error over test days)', round(float(recon.mean()), 4), round(float(pred.mean()), 4), float(err), 'Pass' if err < 1e-3 else 'Fail'))
print('Maximum absolute error:', err)

## V4: LKR effects add up
Each reason has an LKR effect = W x B x K x (occupancy points) / 100. The effect of the typical day plus all reasons must equal the demand effect, W x (demand price - B).

In [ ]:
B = 12000
typ_eff = W * B * K * (base_value - NORMAL) / 100
worst = 0
for i in range(len(test)):
    reasons = W * B * K * sv[i].sum() / 100
    demand_effect = W * (B * (1 + K * (pred[i] - NORMAL) / 100) - B)
    worst = max(worst, abs(typ_eff + reasons - demand_effect))
checks.append(('V4', 'Typical-day effect + reasons = demand effect (max error in LKR)', 0, round(worst, 6), worst, 'Pass' if worst < 0.01 else 'Fail'))
print('Maximum difference (LKR):', worst)

## V5: Revenue simulation recalculated independently
Simulated occupancy = real occupancy x (price / base rate) ^ (-e), capped at 100. RevPAR = price x occupancy / 100. The result is compared with `results/revenue_simulation.csv` from Notebook 3.

In [ ]:
saved = pd.read_csv(f'{RESULTS}/revenue_simulation.csv')
test['pred'] = np.clip(xgb.predict(test[FEATURES]), 0, 100)
test['ai'] = [pe.recommend_price(o, b, c) for o, b, c in zip(test.pred, test.Property_Base_Rate_LKR, test.Competitor_Avg_Price_LKR)]
base = test.Property_Base_Rate_LKR
strat = {'Fixed price (base rate)': base, 'Copy competitor': test.Competitor_Avg_Price_LKR, 'AI price': test.ai}
ok = True
rows = []
for e in sorted(saved.elasticity_e.unique()):
    for name, p in strat.items():
        occ = np.minimum(test.Historical_Occupancy_Rate * (p / base) ** (-e), 100)
        mine = round((p * occ / 100).mean())
        theirs = int(saved[(saved.elasticity_e == e) & (saved.strategy == name)].avg_RevPAR_LKR.iloc[0])
        rows.append((e, name, mine, theirs, mine - theirs))
        ok &= abs(mine - theirs) <= 1
v5 = pd.DataFrame(rows, columns=['e', 'strategy', 'recalculated RevPAR', 'saved RevPAR', 'difference'])
checks.append(('V5', 'Revenue simulation (15 values recalculated)', len(v5), int((v5.difference.abs() <= 1).sum()), int(v5.difference.abs().max()), 'Pass' if ok else 'Fail'))
# identity: at e = 1 every strategy earns base rate x real occupancy
ident = (base * test.Historical_Occupancy_Rate / 100).mean()
r1 = saved[saved.elasticity_e == 1.0].avg_RevPAR_LKR.unique()
checks.append(('V5', 'At e = 1 all strategies equal base rate x occupancy', round(ident), int(r1[0]), abs(round(ident) - int(r1[0])), 'Pass' if len(r1) == 1 and abs(round(ident) - int(r1[0])) <= 1 else 'Fail'))
v5

## Summary table (copy into the appendix)

In [ ]:
out = pd.DataFrame(checks, columns=['id', 'check', 'by hand / expected', 'library / system', 'difference', 'result'])
out.to_csv(f'{RESULTS}/formula_validation.csv', index=False)
print(out.result.value_counts().to_dict())
out